# Phase 3 — Duck Curve Analysis

This notebook estimates ERCOT summer net-load shape and duck-curve depth from hourly load and monthly renewable generation.

In [1]:

import subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
PROJECT_ROOT=Path.cwd()
if PROJECT_ROOT.name=='notebooks': PROJECT_ROOT=PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0,str(PROJECT_ROOT))
CHROME_PATH=PROJECT_ROOT/'.kaleido_chrome/chrome-mac-arm64/Google Chrome for Testing.app/Contents/MacOS/Google Chrome for Testing'
CHART_DIR=PROJECT_ROOT/'outputs/charts'; CHART_DIR.mkdir(parents=True, exist_ok=True)
def apply_chart_standard(fig,title,xaxis_title,yaxis_title,source='Source: ERCOT hourly load; EIA generation; author calculations'):
    fig.update_layout(title=dict(text=title,font=dict(size=18,family='Arial')),font=dict(family='Arial',size=12),xaxis_title=xaxis_title,yaxis_title=yaxis_title,plot_bgcolor='white',paper_bgcolor='white',hovermode='x unified',margin=dict(l=70,r=50,t=90,b=80))
    fig.update_xaxes(showgrid=True,gridcolor='#F5F5F5',title_font=dict(size=14)); fig.update_yaxes(showgrid=True,gridcolor='#F5F5F5',title_font=dict(size=14))
    fig.add_annotation(xref='paper',yref='paper',x=1.0,y=-0.14,text=source,showarrow=False,font=dict(size=10,color='gray'),xanchor='right')
    return fig
def save_chart(fig,filename):
    html_path=CHART_DIR/f'{filename}.html'; png_path=CHART_DIR/f'{filename}.png'
    fig.write_html(html_path); print(f'Saved: {html_path.relative_to(PROJECT_ROOT)}')
    subprocess.run([str(CHROME_PATH),'--headless=new','--disable-gpu','--no-sandbox','--disable-dev-shm-usage',f'--screenshot={png_path}','--window-size=1200,700',f'file://{html_path}'],check=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True)
    print(f'Saved: {png_path.relative_to(PROJECT_ROOT)}')
duck_df=pd.read_csv(PROJECT_ROOT/'data/processed/duck_curve.csv')
print(f'Loaded duck curve dataset: {duck_df.shape}')
print(duck_df.head(10).to_string(index=False))
print('Chart export method: Plotly HTML + approved headless Chrome PNG screenshot')


Loaded duck curve dataset: (720, 10)
 year  hour   season  avg_load_mw  avg_wind_mw  avg_solar_mw  net_load_mw  season_min_net_load  evening_peak_net_load  duck_curve_depth_mw
 2015     0 shoulder 34109.379963  5213.157109     66.509857 28829.712998          23642.75851           37620.215376         13977.456866
 2015     1 shoulder 31798.420263  5213.157109     66.509857 26518.753297          23642.75851           37620.215376         13977.456866
 2015     2 shoulder 30293.043464  5213.157109     66.509857 25013.376498          23642.75851           37620.215376         13977.456866
 2015     3 shoulder 29371.643789  5213.157109     66.509857 24091.976824          23642.75851           37620.215376         13977.456866
 2015     4 shoulder 28922.425475  5213.157109     66.509857 23642.758510          23642.75851           37620.215376         13977.456866
 2015     5 shoulder 29102.400587  5213.157109     66.509857 23822.733622          23642.75851           37620.215376         139

In [2]:

print('DUCK CURVE ANALYSIS')
print('='*50)
duck=duck_df[duck_df['season']=='summer'].copy()
print('Duck curve depth by year (summer, GW):')
for year in sorted(duck['year'].unique()):
    depth=duck[duck['year']==year]['duck_curve_depth_mw'].max()
    print(f'  {year}: {depth/1000:.1f} GW')
depth_2019=duck[duck['year']==2019]['duck_curve_depth_mw'].max()
depth_2024=duck[duck['year']==2024]['duck_curve_depth_mw'].max()
print(f'\nDuck curve deepened {(depth_2024-depth_2019)/1000:.1f} GW from 2019 to 2024')
print(f'This means flexible resources must cover {(depth_2024-depth_2019)/1000:.1f} GW more average summer evening ramp depth than in 2019 under this monthly-renewables approximation')
print('\nMethod note: monthly wind and solar generation are converted to average hourly output, so this is a structural trend indicator rather than an operational dispatch model.')


DUCK CURVE ANALYSIS
Duck curve depth by year (summer, GW):
  2015: 24.5 GW
  2016: 23.8 GW
  2017: 23.5 GW
  2018: 25.3 GW
  2019: 24.7 GW
  2020: 25.7 GW
  2021: 23.9 GW
  2022: 26.3 GW
  2023: 27.4 GW
  2024: 25.3 GW

Duck curve deepened 0.5 GW from 2019 to 2024
This means flexible resources must cover 0.5 GW more average summer evening ramp depth than in 2019 under this monthly-renewables approximation

Method note: monthly wind and solar generation are converted to average hourly output, so this is a structural trend indicator rather than an operational dispatch model.


In [3]:

years_to_show=[2015,2017,2019,2021,2023,2024]
colors=px.colors.sequential.Viridis[:len(years_to_show)]
fig=go.Figure()
for year,color in zip(years_to_show,colors):
    yr_data=duck[duck['year']==year].sort_values('hour')
    fig.add_trace(go.Scatter(x=yr_data['hour'], y=yr_data['net_load_mw']/1000, name=str(year), line=dict(color=color,width=2.5), mode='lines'))
fig.add_annotation(x=13, y=duck[duck['year']==2024]['net_load_mw'].min()/1000 - 1, text='Midday trough reflects<br>rising solar output', showarrow=True, arrowhead=2, bgcolor='white', bordercolor='gray')
fig=apply_chart_standard(fig,'ERCOT Duck Curve Evolution 2015-2024 (Summer Average Day)<br><sub>Net Load = Total Load minus Wind and Solar</sub>','Hour of Day','Net Load (GW)')
fig.update_xaxes(tickvals=list(range(0,24,2)), ticktext=[f'{h:02d}:00' for h in range(0,24,2)])
save_chart(fig,'chart_4a_duck_curve_evolution')
fig.show()


Saved: outputs/charts/chart_4a_duck_curve_evolution.html


Saved: outputs/charts/chart_4a_duck_curve_evolution.png


In [4]:

depth = duck.groupby('year', as_index=False)['duck_curve_depth_mw'].max()
depth['duck_curve_depth_gw']=depth['duck_curve_depth_mw']/1000
print('Duck curve depth table:')
print(depth.to_string(index=False))
fig=go.Figure(go.Bar(x=depth['year'], y=depth['duck_curve_depth_gw'], marker_color='#4CAF50', text=[f'{v:.1f} GW' for v in depth['duck_curve_depth_gw']], textposition='outside'))
fig=apply_chart_standard(fig,'ERCOT Summer Duck Curve Depth by Year','Year','Duck Curve Depth (GW)')
fig.update_yaxes(range=[0, max(depth['duck_curve_depth_gw'])*1.20])
save_chart(fig,'chart_4b_duck_curve_depth')
fig.show()


Duck curve depth table:
 year  duck_curve_depth_mw  duck_curve_depth_gw
 2015         24480.758963            24.480759
 2016         23806.479155            23.806479
 2017         23526.897416            23.526897
 2018         25326.986932            25.326987
 2019         24745.800663            24.745801
 2020         25747.259858            25.747260
 2021         23867.898034            23.867898
 2022         26347.655817            26.347656
 2023         27439.802898            27.439803
 2024         25268.065118            25.268065
Saved: outputs/charts/chart_4b_duck_curve_depth.html


Saved: outputs/charts/chart_4b_duck_curve_depth.png
